# Bước 03-1: Đặc trưng thời gian, Lag và Rolling
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

Notebook này đọc các tập split từ bước 02 và sinh:
- Đặc trưng lịch + tuần hoàn (từ timestamp)
- Đặc trưng lag và rolling (từ target, có backward context)

Kết quả ghi ra `data/model/v3/03_1_features_time/`.

> ### Lưu ý về RAM trước khi chạy
>
> Notebook xử lý nhiều triệu dòng dữ liệu. Trước khi chạy: đóng kernel
> của các notebook khác trong VSCode. Mỗi kernel giữ vài GB và không tự nhả
> sau khi chạy xong.
>
> Notebook đã ghi file và `gc.collect()` ngay sau mỗi tập để không giữ
> nhiều DataFrame lớn cùng lúc.

## 2. Import thư viện và khai báo tham số

In [5]:
import gc
import json
import os

import numpy as np
import pandas as pd

# ── Tham so dac trung ──
VERSION = 'v3'
EXPECTED_FREQ_MINUTES = 15
LAGS = (1, 4, 96)                    # 15 phut, 1 gio, 24 gio
ROLLING_WINDOWS = (4, 12, 96)        # 1 gio, 3 gio, 24 gio

CATEGORICAL_COLS = (
    'site_id', 'campus_name', 'location_name', 'site_metric', 'panel',
    'inverter', 'optimizers', 'weather_join_method',
    'weather_condition', 'weather_description',
)

# ── Ten cot ──
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'

# ── Ban do mua Nam ban cau ──
SOUTHERN_SEASON_MAP = {
    12: 'summer', 1: 'summer', 2: 'summer',
    3: 'autumn', 4: 'autumn', 5: 'autumn',
    6: 'winter', 7: 'winter', 8: 'winter',
    9: 'spring', 10: 'spring', 11: 'spring',
}
SEASON_CODE_MAP = {'summer': 0, 'autumn': 1, 'winter': 2, 'spring': 3}

SPLIT_DIR = '../../data/model/v3/02_split'
OUTPUT_DIR = '../../data/model/v3/03_1_features_time'

print("Da import thu vien va khai bao tham so.")
print(f"- Version         : {VERSION}")
print(f"- Tan suat ky vong: {EXPECTED_FREQ_MINUTES} phut")
print(f"- Lags            : {LAGS}")
print(f"- Rolling windows : {ROLLING_WINDOWS}")
print(f"- Doc split tu    : {SPLIT_DIR}")
print(f"- Ghi dac trung ra: {OUTPUT_DIR}")

Da import thu vien va khai bao tham so.
- Version         : v3
- Tan suat ky vong: 15 phut
- Lags            : (1, 4, 96)
- Rolling windows : (4, 12, 96)
- Doc split tu    : ../../data/model/v3/02_split
- Ghi dac trung ra: ../../data/model/v3/03_1_features_time


## 3. Hàm đọc/ghi parquet

In [6]:
def require_columns(df, columns):
    """Bao loi som neu thieu cot bat buoc."""
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")


def read_parquet(path):
    """Doc parquet, kiem tra cot bat buoc, ep kieu timestamp va sap xep."""
    df = pd.read_parquet(path)
    require_columns(df, [TIMESTAMP_COL, SITE_COL, TARGET_COL])
    df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors='coerce')
    return df.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)


def write_parquet(df, path):
    """Ghi parquet, tu tao thu muc neu chua co."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)
    return path


print("Da dinh nghia require_columns, read_parquet, write_parquet.")

Da dinh nghia require_columns, read_parquet, write_parquet.


## 4. Đặc trưng thời gian (lịch + tuần hoàn)

Chỉ suy ra từ cột `timestamp`, không đụng tới target nên không có rò rỉ.

In [7]:
def add_time_features(df):
    """Tao dac trung lich va tuan hoan chi tu timestamp."""

    out = df.copy()
    ts = pd.to_datetime(out[TIMESTAMP_COL], errors='coerce')
    minute_of_day = ts.dt.hour * 60 + ts.dt.minute
    day_of_year = ts.dt.dayofyear

    out['minute_of_day'] = minute_of_day
    out['hour_sin'] = np.sin(2 * np.pi * minute_of_day / 1440.0)
    out['hour_cos'] = np.cos(2 * np.pi * minute_of_day / 1440.0)
    out['doy_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    out['doy_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)
    out['month'] = ts.dt.month
    out['day_of_week'] = ts.dt.dayofweek
    out['is_weekend'] = out['day_of_week'].isin([5, 6]).astype('int8')
    out['season'] = out['month'].map(SOUTHERN_SEASON_MAP).astype('string')
    out['season_code'] = out['season'].map(SEASON_CODE_MAP).astype('Int64')

    # Some marts use hour=-1 to represent the previous-hour bucket. For model
    # timestamp features, keep both raw and model-safe version.
    if 'hour' in out.columns:
        out['hour_bucket_raw'] = pd.to_numeric(out['hour'], errors='coerce')
        out['hour_bucket_model'] = out['hour_bucket_raw'].replace(-1, 23)

    return out


print("Da dinh nghia add_time_features.")

Da dinh nghia add_time_features.


## 5. Mặt nạ lịch sử liên tục

Nếu cửa sổ lịch sử vắt qua chỗ đứt gãy thời gian thì giá trị lag/rolling
tính ra vô nghĩa. Hàm này trả về `True` chỉ khi `window_steps` khoảng thời gian
liền trước đều đúng 15 phút.

In [8]:
def continuous_history_mask(group, window_steps):
    """True khi window_steps khoang thoi gian lien truoc deu lien tuc."""
    diffs = group[TIMESTAMP_COL].diff().dt.total_seconds().div(60.0)
    is_expected_gap = diffs.eq(EXPECTED_FREQ_MINUTES)
    return (
        is_expected_gap.rolling(window_steps, min_periods=window_steps)
        .sum()
        .eq(window_steps)
        .fillna(False)
    )


print("Da dinh nghia continuous_history_mask.")

Da dinh nghia continuous_history_mask.


## 6. Đặc trưng Lag và Rolling

Mọi đặc trưng suy từ target đều được `shift`. Rolling luôn dùng target **đã shift(1)**.
Giá trị bị gán `NaN` nếu cửa sổ lịch sử vắt qua chỗ đứt gãy thời gian.

In [9]:
def add_lag_rolling_features(df):
    """Tao dac trung lag/rolling an toan ve ro ri.

    Moi dac trung suy tu target deu duoc shift. Rolling dung target da shift(1).
    Gia tri bi gan NaN neu cua so lich su vat qua cho dut gay thoi gian.
    """

    out = df.copy()
    out = out.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)
    grouped = out.groupby(SITE_COL, group_keys=False, observed=True)

    for lag in LAGS:
        col = f'lag_{lag}'
        out[col] = grouped[TARGET_COL].shift(lag)
        valid = grouped.apply(continuous_history_mask, window_steps=lag)
        out.loc[~valid.to_numpy(), col] = np.nan

    shifted_target = grouped[TARGET_COL].shift(1)

    for window in ROLLING_WINDOWS:
        valid = grouped.apply(continuous_history_mask, window_steps=window).to_numpy()
        rolling = shifted_target.groupby(out[SITE_COL]).rolling(window, min_periods=window)
        feature_map = {
            f'rolling_mean_{window}': rolling.mean(),
            f'rolling_std_{window}': rolling.std(),
            f'rolling_min_{window}': rolling.min(),
            f'rolling_max_{window}': rolling.max(),
        }
        for col, values in feature_map.items():
            out[col] = values.reset_index(level=0, drop=True)
            out.loc[~valid, col] = np.nan

    target_feature_cols = [
        col for col in out.columns
        if col.startswith('lag_') or col.startswith('rolling_')
    ]
    out['has_complete_history_features'] = ~out[target_feature_cols].isna().any(axis=1)
    return out


print("Da dinh nghia add_lag_rolling_features.")
print(f"So dac trung target se tao: {len(LAGS)} lag + {len(ROLLING_WINDOWS) * 4} rolling = "
      f"{len(LAGS) + len(ROLLING_WINDOWS) * 4} cot")

Da dinh nghia add_lag_rolling_features.
So dac trung target se tao: 3 lag + 12 rolling = 15 cot


## 7. Xây đặc trưng với backward context

Hàm trung tâm: nối `context_df` vào đầu `target_df`, tính đặc trưng trên phần gộp,
rồi **chỉ xuất các dòng thuộc `target_df`**.

Phiên bản rút gọn: chỉ gọi `add_time_features` và `add_lag_rolling_features`.
Metadata và weather domain xử lý ở các bước sau (03-2 và 03-3).

In [10]:
def build_features_with_backward_context(context_df, target_df, output_role):
    """Tao dac trung cho cac dong target, co the kem lich su tu split truoc do.

    Phien ban rut gon: chi goi add_time_features va add_lag_rolling_features.
    Metadata va weather domain xu ly o cac buoc sau (03_2 va 03_3).
    """
    target = target_df.copy()
    target['_feature_export_row'] = True
    target['_feature_output_role'] = output_role

    frames = []
    if context_df is not None and len(context_df):
        context = context_df.copy()
        context['_feature_export_row'] = False
        context['_feature_output_role'] = 'history_context'
        frames.append(context)
    frames.append(target)

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined[TIMESTAMP_COL] = pd.to_datetime(combined[TIMESTAMP_COL], errors='coerce')
    combined = combined.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)

    features = add_time_features(combined)
    features = add_lag_rolling_features(features)

    export = features[features['_feature_export_row'].astype(bool)].copy()
    export = export.drop(columns=['_feature_export_row'])
    return export.reset_index(drop=True)


print("Da dinh nghia build_features_with_backward_context (chi time + lag/rolling).")

Da dinh nghia build_features_with_backward_context (chi time + lag/rolling).


## 8. Đọc các tập đã split từ Notebook 02

In [11]:
development = read_parquet(f'{SPLIT_DIR}/development/{VERSION}_development.parquet')
test = read_parquet(f'{SPLIT_DIR}/test/{VERSION}_test.parquet')
train_alias = read_parquet(f'{SPLIT_DIR}/train/{VERSION}_train.parquet')
val_alias = read_parquet(f'{SPLIT_DIR}/val/{VERSION}_val.parquet')

print("Da doc cac tap split tu Notebook 02:")
for name, df in [('development', development), ('test', test),
                 ('train_alias', train_alias), ('val_alias', val_alias)]:
    print(f"- {name:<12}: {len(df)} dong  {df[TIMESTAMP_COL].min()} -> {df[TIMESTAMP_COL].max()}")
display(development.head(3))

Da doc cac tap split tu Notebook 02:
- development : 2273970 dong  2020-01-01 00:15:00 -> 2021-12-18 09:15:00
- test        : 510468 dong  2021-12-18 09:30:00 -> 2022-04-23 23:45:00
- train_alias : 1791894 dong  2020-01-01 00:15:00 -> 2021-08-20 19:45:00
- val_alias   : 482076 dong  2021-08-20 20:00:00 -> 2021-12-18 09:15:00


,timestamp,gen_id,site_id,geo_id,date_id,time_id,is_dst_repeat,full_date,year,month,...,month_model,day_of_year,season_model,weather_is_observed,is_daylight,outlier_group,v3_holdout_split,v3_test_start_timestamp,v3_split_strategy,v3_n_time_series_splits
0,2020-01-01 00:15:00,1,1,1,20200101,15,0,2020-01-01,2020,1,...,1,1,summer,True,False,normal,development,2021-12-18 09:30:00,expanding,5
1,2020-01-01 00:30:00,11,1,1,20200101,30,0,2020-01-01,2020,1,...,1,1,summer,True,False,normal,development,2021-12-18 09:30:00,expanding,5
2,2020-01-01 00:45:00,21,1,1,20200101,45,0,2020-01-01,2020,1,...,1,1,summer,True,False,normal,development,2021-12-18 09:30:00,expanding,5


## 9. Sinh đặc trưng thời gian cho development và test

- `development`: không cần context, nó là dữ liệu sớm nhất
- `test`: dùng **toàn bộ `development`** làm backward context, sau đó chỉ xuất phần `test`

In [12]:
development_time = build_features_with_backward_context(
    context_df=None, target_df=development, output_role='development')

test_time = build_features_with_backward_context(
    context_df=development, target_df=test, output_role='test')

write_parquet(development_time, f'{OUTPUT_DIR}/{VERSION}_development_time.parquet')
write_parquet(test_time, f'{OUTPUT_DIR}/{VERSION}_test_time.parquet')
n_dev, c_dev = len(development_time), development_time.shape[1]
n_test, c_test = len(test_time), test_time.shape[1]
del development_time, test_time, development, test
gc.collect()

print("Da sinh dac trung thoi gian cho development va test (da ghi file va giai phong RAM).")
print(f"- development: {n_dev} dong x {c_dev} cot")
print(f"- test       : {n_test} dong x {c_test} cot")

Da sinh dac trung thoi gian cho development va test (da ghi file va giai phong RAM).
- development: 2273970 dong x 103 cot
- test       : 510468 dong x 104 cot


## 10. Sinh đặc trưng thời gian cho train/val

`val_alias` dùng `train_alias` làm backward context.

In [13]:
train_time = build_features_with_backward_context(
    context_df=None, target_df=train_alias, output_role='train_alias')

val_time = build_features_with_backward_context(
    context_df=train_alias, target_df=val_alias, output_role='val_alias')

write_parquet(train_time, f'{OUTPUT_DIR}/{VERSION}_train_time.parquet')
write_parquet(val_time, f'{OUTPUT_DIR}/{VERSION}_val_time.parquet')
n_tr, c_tr = len(train_time), train_time.shape[1]
n_va, c_va = len(val_time), val_time.shape[1]
del train_alias, val_alias, train_time, val_time
gc.collect()

print("Da sinh dac trung thoi gian cho train/val (da ghi file va giai phong RAM).")
print(f"- train_alias: {n_tr} dong x {c_tr} cot")
print(f"- val_alias  : {n_va} dong x {c_va} cot")

Da sinh dac trung thoi gian cho train/val (da ghi file va giai phong RAM).
- train_alias: 1791894 dong x 104 cot
- val_alias  : 482076 dong x 104 cot


## 11. Sinh đặc trưng thời gian cho 5 fold cross-validation

Với mỗi fold:
- `fold_train`: không cần context
- `fold_val`: dùng chính `fold_train` của nó làm backward context

Nhờ vậy mỗi fold là một thí nghiệm khép kín.

In [14]:
folds_dir = f'{SPLIT_DIR}/time_series_folds'
feature_folds_dir = f'{OUTPUT_DIR}/time_series_folds'

fold_train_paths = sorted(
    p for p in os.listdir(folds_dir) if p.startswith('fold_') and p.endswith('_train.parquet')
)
print(f"Tim thay {len(fold_train_paths)} fold trong {folds_dir}\n")

for fold_train_name in fold_train_paths:
    fold_name = fold_train_name.replace('_train.parquet', '')
    fold_val_name = f'{fold_name}_val.parquet'
    fold_val_path = f'{folds_dir}/{fold_val_name}'
    if not os.path.exists(fold_val_path):
        raise FileNotFoundError(f"Missing fold validation parquet: {fold_val_path}")

    fold_train = read_parquet(f'{folds_dir}/{fold_train_name}')
    fold_val = read_parquet(fold_val_path)

    fold_train_time = build_features_with_backward_context(
        context_df=None, target_df=fold_train, output_role=f'{fold_name}_train')
    fold_val_time = build_features_with_backward_context(
        context_df=fold_train, target_df=fold_val, output_role=f'{fold_name}_val')

    write_parquet(fold_train_time, f'{feature_folds_dir}/{fold_name}_train_time.parquet')
    write_parquet(fold_val_time, f'{feature_folds_dir}/{fold_name}_val_time.parquet')

    print(f"{fold_name}: train {len(fold_train_time)} dong | "
          f"val {len(fold_val_time)} dong | {fold_train_time.shape[1]} cot")

    del fold_train, fold_val, fold_train_time, fold_val_time
    gc.collect()

print(f"\nDa ghi {len(fold_train_paths) * 2} file dac trung fold vao {feature_folds_dir}")

Tim thay 5 fold trong ../../data/model/v3/02_split/time_series_folds

fold_1: train 132523 dong | val 289097 dong | 105 cot
fold_2: train 421620 dong | val 417729 dong | 105 cot
fold_3: train 839349 dong | val 470469 dong | 105 cot
fold_4: train 1309818 dong | val 482076 dong | 105 cot
fold_5: train 1791894 dong | val 482076 dong | 105 cot

Da ghi 10 file dac trung fold vao ../../data/model/v3/03_1_features_time/time_series_folds


## 12. Hoàn tất

In [15]:
print("Hoan tat buoc 03-1: Dac trung thoi gian, lag va rolling.")
print(f"Tat ca file da ghi vao: {OUTPUT_DIR}")

Hoan tat buoc 03-1: Dac trung thoi gian, lag va rolling.
Tat ca file da ghi vao: ../../data/model/v3/03_1_features_time
